# 05 — SimCLR Fine-tuning & Final Comparison

Compare three transfer strategies built on the **same SimCLR backbone**
(`checkpoints/simclr_backbone.pth`) against the **supervised baseline** from nb02.

| Strategy | Encoder | Classification head |
|---|---|---|
| **Linear Probe** | frozen | 1 Linear layer trained |
| **Full Fine-tuning** | unfrozen (low lr) | 1 Linear layer trained |
| **Supervised Baseline** *(nb02)* | ImageNet pre-trained, fully unfrozen | same head |


## 0 — Setup & Config

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from src.ssl import SimCLR
from src.utils import (
    set_seed, compute_metrics, fig_path,
    save_figure, save_results_csv,
    plot_confusion_matrix, plot_roc_curve,
    plot_training_curves, log,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
CFG = {
    # ── paths ────────────────────────────────────────────────────────────────
    'data_root'       : '../datasets/raw',          # contains AI/ and Human/
    'backbone_path'   : '../checkpoints/simclr_backbone.pth',
    'checkpoint_dir'  : '../checkpoints',
    'reports_dir'     : '../reports',
    'nb'              : '05_simclr_finetune',        # used by fig_path()

    # ── data ─────────────────────────────────────────────────────────────────
    'image_size'      : 224,
    'batch_size'      : 64,
    'num_workers'     : 2,
    'seed'            : 42,
    'split_ratios'    : [0.70, 0.15, 0.15],         # same as nb02

    # ── linear probe ─────────────────────────────────────────────────────────
    'lp_epochs'       : 20,
    'lp_lr'           : 1e-3,

    # ── full fine-tuning ─────────────────────────────────────────────────────
    'ft_epochs'       : 30,
    'ft_lr_head'      : 1e-3,
    'ft_lr_backbone'  : 1e-5,   # much smaller to avoid catastrophic forgetting

    # ── misc ──────────────────────────────────────────────────────────────────
    'proj_dim'        : 128,
}

set_seed(CFG['seed'])
os.makedirs(CFG['checkpoint_dir'], exist_ok=True)
os.makedirs(CFG['reports_dir'],    exist_ok=True)


## 1 — Data Loaders

In [ ]:
# ── transforms ───────────────────────────────────────────────────────────────
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(CFG['image_size'], scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(int(CFG['image_size'] * 1.14)),
    transforms.CenterCrop(CFG['image_size']),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])


In [ ]:
# ── stratified split — same seed as nb02 for a fair comparison ───────────────
full_dataset = datasets.ImageFolder(CFG['data_root'])
print(f'Total images: {len(full_dataset)} | Classes: {full_dataset.classes}')

targets = np.array(full_dataset.targets)
indices = np.arange(len(full_dataset))

train_ratio, val_ratio, test_ratio = CFG['split_ratios']
trainval_idx, test_idx = train_test_split(
    indices, test_size=test_ratio, stratify=targets, random_state=CFG['seed']
)
train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=val_ratio / (train_ratio + val_ratio),
    stratify=targets[trainval_idx],
    random_state=CFG['seed'],
)
print(f'Split sizes — train: {len(train_idx)} | val: {len(val_idx)} | test: {len(test_idx)}')

def _make_loader(idx, transform, shuffle=False):
    ds = datasets.ImageFolder(CFG['data_root'], transform=transform)
    return DataLoader(
        Subset(ds, idx),
        batch_size=CFG['batch_size'],
        shuffle=shuffle,
        num_workers=CFG['num_workers'],
        pin_memory=True,
    )

train_loader = _make_loader(train_idx, train_transform, shuffle=True)
val_loader   = _make_loader(val_idx,   eval_transform,  shuffle=False)
test_loader  = _make_loader(test_idx,  eval_transform,  shuffle=False)


## 2 — Load Pre-trained SimCLR Backbone

In [ ]:
def load_backbone(path: str) -> nn.Module:
    """Load the ResNet18 encoder saved by nb04.

    nb04 saves ``{'encoder_state_dict': ..., 'proj_dim': ...}``.
    Also supports legacy flat dicts with ``encoder.xxx`` keys.
    """
    simclr = SimCLR(proj_dim=CFG['proj_dim'])
    state = torch.load(path, map_location='cpu')

    # nb04 format: dict with encoder_state_dict key
    if 'encoder_state_dict' in state:
        enc_state = state['encoder_state_dict']
        log(f'Backbone loaded from {path}  (nb04 format)')
    elif any(k.startswith('encoder.') for k in state):
        enc_state = {k[len('encoder.'):]: v
                     for k, v in state.items() if k.startswith('encoder.')}
        log(f'Backbone loaded from {path}  (legacy encoder.xxx format)')
    else:
        enc_state = state
        log(f'Backbone loaded from {path}  (bare state_dict)')

    simclr.encoder.load_state_dict(enc_state, strict=False)
    return simclr.encoder   # bare ResNet18, fc = nn.Identity()


backbone = load_backbone(CFG['backbone_path'])
print(backbone)

## 3 — Linear Probing

The encoder is **completely frozen**; only the newly added linear head is trained.
This measures how well SimCLR representations are linearly separable.

In [ ]:
class LinearProbe(nn.Module):
    """Frozen encoder + trainable linear head."""

    def __init__(self, encoder: nn.Module, feat_dim: int = 512, num_classes: int = 2):
        super().__init__()
        self.encoder = encoder
        self.head    = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():   # encoder is frozen — no gradient through it
            h = self.encoder(x)
        return self.head(h)


def _train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct    += (logits.argmax(1) == y).sum().item()
        total      += x.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def _eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits     = model(x)
        loss       = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        correct    += (logits.argmax(1) == y).sum().item()
        total      += x.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def _collect_preds(model, loader):
    model.eval()
    all_true, all_pred, all_prob = [], [], []
    for x, y in loader:
        x = x.to(DEVICE)
        logits = model(x)
        probs  = torch.softmax(logits, dim=1)
        all_true.extend(y.cpu().numpy())
        all_pred.extend(probs.argmax(1).cpu().numpy())
        all_prob.extend(probs[:, 1].cpu().numpy())   # prob(fake)
    return all_true, all_pred, all_prob


In [ ]:
# ── train linear probe ───────────────────────────────────────────────────────
lp_model = LinearProbe(backbone, feat_dim=512, num_classes=2).to(DEVICE)

# freeze encoder
for p in lp_model.encoder.parameters():
    p.requires_grad_(False)

lp_optimizer = optim.Adam(lp_model.head.parameters(), lr=CFG['lp_lr'])
lp_scheduler = optim.lr_scheduler.CosineAnnealingLR(lp_optimizer, T_max=CFG['lp_epochs'])
criterion    = nn.CrossEntropyLoss()

lp_history, best_lp_acc, best_lp_epoch = [], 0.0, 0

for epoch in range(1, CFG['lp_epochs'] + 1):
    tr_loss, tr_acc = _train_epoch(lp_model, train_loader, criterion, lp_optimizer)
    vl_loss, vl_acc = _eval_epoch (lp_model, val_loader,   criterion)
    lp_scheduler.step()

    lp_history.append({'epoch': epoch, 'train_loss': tr_loss,
                        'val_accuracy': vl_acc, 'val_loss': vl_loss})

    if vl_acc > best_lp_acc:
        best_lp_acc  = vl_acc
        best_lp_epoch = epoch
        torch.save(lp_model.state_dict(),
                   os.path.join(CFG['checkpoint_dir'], 'simclr_linear_probe.pth'))

    print(f'Epoch {epoch:3d}/{CFG["lp_epochs"]} | '
          f'train_loss={tr_loss:.4f}  val_acc={vl_acc:.4f}')

print(f'\nBest val acc: {best_lp_acc:.4f} at epoch {best_lp_epoch}')


In [ ]:
# ── evaluate linear probe on test set ────────────────────────────────────────
lp_model.load_state_dict(
    torch.load(os.path.join(CFG['checkpoint_dir'], 'simclr_linear_probe.pth'),
               map_location=DEVICE))

y_true_lp, y_pred_lp, y_prob_lp = _collect_preds(lp_model, test_loader)
lp_metrics = compute_metrics(y_true_lp, y_pred_lp, y_prob_lp)
lp_metrics['method'] = 'SimCLR — Linear Probe'
print(pd.Series({k: v for k, v in lp_metrics.items()
                 if k in ['accuracy','f1','auc']}))


## 4 — Full Fine-tuning

All backbone weights are **unfrozen** but optimised with a very small learning rate
(`ft_lr_backbone=1e-5`) to avoid catastrophic forgetting of the SSL features.

In [ ]:
def build_finetuned_model(backbone_path: str) -> nn.Module:
    """Fresh SimCLR backbone (re-loaded) + linear head, all parameters trainable."""
    enc = load_backbone(backbone_path)

    class FullModel(nn.Module):
        def __init__(self, encoder, feat_dim=512, num_classes=2):
            super().__init__()
            self.encoder = encoder
            self.head    = nn.Linear(feat_dim, num_classes)
        def forward(self, x):
            return self.head(self.encoder(x))

    return FullModel(enc)


ft_model = build_finetuned_model(CFG['backbone_path']).to(DEVICE)

# two param groups: backbone (low lr) + head (higher lr)
ft_optimizer = optim.Adam([
    {'params': ft_model.encoder.parameters(), 'lr': CFG['ft_lr_backbone']},
    {'params': ft_model.head.parameters(),    'lr': CFG['ft_lr_head']},
])
ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    ft_optimizer, T_max=CFG['ft_epochs'])

ft_history, best_ft_acc, best_ft_epoch = [], 0.0, 0

for epoch in range(1, CFG['ft_epochs'] + 1):
    tr_loss, tr_acc = _train_epoch(ft_model, train_loader, criterion, ft_optimizer)
    vl_loss, vl_acc = _eval_epoch (ft_model, val_loader,   criterion)
    ft_scheduler.step()

    ft_history.append({'epoch': epoch, 'train_loss': tr_loss,
                        'val_accuracy': vl_acc, 'val_loss': vl_loss})

    if vl_acc > best_ft_acc:
        best_ft_acc  = vl_acc
        best_ft_epoch = epoch
        torch.save(ft_model.state_dict(),
                   os.path.join(CFG['checkpoint_dir'], 'simclr_finetuned.pth'))

    print(f'Epoch {epoch:3d}/{CFG["ft_epochs"]} | '
          f'train_loss={tr_loss:.4f}  val_acc={vl_acc:.4f}')

print(f'\nBest val acc: {best_ft_acc:.4f} at epoch {best_ft_epoch}')


In [ ]:
# ── evaluate full fine-tuning on test set ────────────────────────────────────
ft_model.load_state_dict(
    torch.load(os.path.join(CFG['checkpoint_dir'], 'simclr_finetuned.pth'),
               map_location=DEVICE))

y_true_ft, y_pred_ft, y_prob_ft = _collect_preds(ft_model, test_loader)
ft_metrics = compute_metrics(y_true_ft, y_pred_ft, y_prob_ft)
ft_metrics['method'] = 'SimCLR — Full Fine-tuning'
print(pd.Series({k: v for k, v in ft_metrics.items()
                 if k in ['accuracy','f1','auc']}))


## 5 — Final Comparison Table

In [ ]:
# ── supervised baseline (nb02 results) ───────────────────────────────────────
# nb02 saves as '02_baseline_results.csv' (not '02_supervised_baseline_results.csv')
_baseline_csv = os.path.join(CFG['reports_dir'], '02_baseline_results.csv')

if os.path.exists(_baseline_csv):
    baseline_row = pd.read_csv(_baseline_csv).iloc[0].to_dict()
    baseline_row['method'] = 'Supervised Baseline (nb02)'
else:
    baseline_row = {'method': 'Supervised Baseline (nb02)',
                    'accuracy': None, 'f1': None, 'auc': None}
    log('WARNING: baseline CSV not found — skipping baseline in comparison.')

records = [baseline_row, lp_metrics, ft_metrics]
save_results_csv(
    records,
    os.path.join(CFG['reports_dir'], '05_simclr_finetune_results.csv')
)

summary_cols = ['method', 'accuracy', 'f1', 'auc']
df_summary = pd.DataFrame(records)[summary_cols].copy()
df_summary[['accuracy', 'f1', 'auc']] = df_summary[['accuracy', 'f1', 'auc']].round(4)
df_summary.columns = ['Method', 'Accuracy', 'F1', 'AUC-ROC']
df_summary

## 6 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, history, label in [
    (axes[0], lp_history, 'Linear Probe'),
    (axes[1], ft_history, 'Full Fine-tuning'),
]:
    epochs    = [h['epoch'] for h in history]
    val_accs  = [h['val_accuracy'] for h in history]
    tr_losses = [h['train_loss']   for h in history]

    ax2 = ax.twinx()
    ax.plot(epochs, tr_losses, label='Train loss', color='steelblue')
    ax2.plot(epochs, val_accs, label='Val accuracy', color='tomato', linestyle='--')

    ax.set_title(f'{label} — Training Dynamics')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Train Loss', color='steelblue')
    ax2.set_ylabel('Val Accuracy', color='tomato')

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='center right')

fig.tight_layout()
save_figure(fig,
    fig_path(CFG['reports_dir'], CFG['nb'], 'training_curves.png'))


## 7 — ROC Curves (all methods)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

# nb02 does NOT save per-sample predictions, only aggregated metrics.
# From CSV we skip baseline ROC (no preds file exists).
log('NOTE: nb02 does not save per-sample predictions — baseline ROC unavailable.')

from sklearn.metrics import roc_curve as _roc

for y_true, y_prob, label in [
    (y_true_lp, y_prob_lp, f'Linear Probe  AUC={lp_metrics["auc"]:.3f}'),
    (y_true_ft, y_prob_ft, f'Full Fine-tune AUC={ft_metrics["auc"]:.3f}'),
]:
    fpr, tpr, _ = _roc(y_true, y_prob)
    ax.plot(fpr, tpr, label=label)

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves — SimCLR Methods'); ax.legend()
fig.tight_layout()
save_figure(fig, fig_path(CFG['reports_dir'], CFG['nb'], 'roc_all_methods.png'))

## 8 — Comparison Bar Chart

In [ ]:
metrics_to_plot = ['Accuracy', 'F1', 'AUC-ROC']
x = np.arange(len(df_summary))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
for i, metric in enumerate(metrics_to_plot):
    bars = ax.bar(x + i * width, df_summary[metric], width, label=metric)
    ax.bar_label(bars, fmt='%.3f', fontsize=8, padding=2)

ax.set_xticks(x + width)
ax.set_xticklabels(df_summary['Method'], rotation=12, ha='right')
ax.set_ylim(0, 1.08)
ax.set_ylabel('Score')
ax.set_title('Supervised Baseline vs. SimCLR Strategies')
ax.legend()
fig.tight_layout()
save_figure(fig, fig_path(CFG['reports_dir'], CFG['nb'], 'comparison_bar.png'))


## 9 — Confusion Matrices

In [ ]:
for y_true, y_pred, label, fname in [
    (y_true_lp, y_pred_lp, 'SimCLR — Linear Probe',      'cm_linear_probe.png'),
    (y_true_ft, y_pred_ft, 'SimCLR — Full Fine-tuning',   'cm_finetuned.png'),
]:
    plot_confusion_matrix(
        y_true, y_pred,
        title=label,
        path=fig_path(CFG['reports_dir'], CFG['nb'], fname),
    )


## 10 — Summary & Observations

In [ ]:
print('=' * 65)
print(f'  {"Method":<30} {"Acc":>6} {"F1":>6} {"AUC":>6}')
print('-' * 65)
for _, row in df_summary.iterrows():
    print(f'  {row["Method"]:<30} {row["Accuracy"]:>6.4f} {row["F1"]:>6.4f} {row["AUC-ROC"]:>6.4f}')
print('=' * 65)

for col in ['Accuracy', 'F1', 'AUC-ROC']:
    best_idx = df_summary[col].idxmax()
    print(f'Best {col}: {df_summary.loc[best_idx, "Method"]}  ({df_summary.loc[best_idx, col]:.4f})')


### Observations

- **Linear Probe** shows how well SimCLR representations are *already* linearly separable without any label signal during pre-training.
- **Full Fine-tuning** typically closes the gap with — or surpasses — the supervised baseline when labeled data is limited;
- The SSL pre-training acts as a strong regulariser: the encoder has seen all the unlabeled images and learned invariances that help with real vs. AI discrimination.
- The baseline (nb02) trains from ImageNet weights with supervision; SimCLR fine-tuning starts from domain-specific self-supervised weights.

Checkpoints saved:
- `checkpoints/simclr_linear_probe.pth`
- `checkpoints/simclr_finetuned.pth`

Results CSV → `reports/05_simclr_finetune_results.csv`
